In [3]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
import glob

In [4]:
def convert_img_to_array(x):
    x = np.asarray(x)
    while x.dtype == object:
        x = np.asarray(x.item() if x.ndim == 0 else x[0])
    return x

In [5]:
regions_df = pd.read_csv('../20260617_darkfield_data_for_NN.csv')
regions_df

,Unnamed: 0,Area,ImageX,ImageY,Intensity_gfp,Array_gfp,Intensity_abs,Array_abs,view
0,0,180.0,5.894444,5.894444,921.655556,[[ 942 892 1020 987 1002 965 939 921 1022...,0.138163,[[ 0.11643387 0.09903811 0.11986743 0.09132...,1
1,1,278.0,9.676259,2953.194245,992.359712,[[ 928 902 924 984 967 1002 970 992 1103...,-0.032486,[[-0.06998746 -0.0208492 -0.03581766 -0.02185...,1
2,2,346.0,186.500000,210.500000,957.112717,[[ 0 0 0 0 0 0 0 0 0...,-0.000368,[[ 0.00000000e+00 0.00000000e+00 0.00000000e...,1
3,3,241.0,217.547718,4.854772,945.892116,[[ 0 0 943 0 0 0 0 0 0...,0.015796,[[-0.00000000e+00 -0.00000000e+00 5.12502994e...,1
4,4,387.0,251.470284,468.180879,1028.870801,[[ 0 0 0 0 0 0 0 0 0...,0.055115,[[ 0.00000000e+00 0.00000000e+00 0.00000000e...,1
...,...,...,...,...,...,...,...,...,...
12829,24,981.0,537.608563,568.349643,1032.898063,[[0 0 0 ... 0 0 0]\n [0 0 0 ... 0 0 0]\n [0 0 ...,0.379113,[[0. 0. 0. ... 0. 0. 0.]\n [0. 0. 0. ... 0. 0....,14
12830,25,1021.0,577.830558,2953.486778,974.387855,[[ 0 0 0 ... 0 0 993]\n [ 0 ...,-0.042277,[[-0. -0. -0. ... -0. ...,14
12831,26,559.0,562.588551,344.057245,1003.220036,[[ 0 0 0 0 0 0 0 0 0...,0.370344,[[0. 0. 0. 0. ...,14
12832,27,576.0,570.802083,143.541667,953.045139,[[ 0 0 0 0 0 0 0 0 0...,0.341923,[[0. 0. 0. 0. ...,14


In [ ]:
class Ds(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label